In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import welch

# -----------------------------
# PARAMETERS
# -----------------------------
FS = 200
WINDOW=200
STEP=1

ALPHA = 1.0   # weight for RMS
BETA  = 1.0   # weight for MDF

# -----------------------------
# LOAD CSV
# -----------------------------
df = pd.read_csv("data/emg_data_600.csv")

# -----------------------------
# FEATURE FUNCTIONS
# -----------------------------
def rms(x):
    return np.sqrt(np.mean(x**2))

def mdf(signal, fs):
    f, Pxx = welch(signal, fs=fs, nperseg=len(signal))
    cumsum = np.cumsum(Pxx)
    return f[np.where(cumsum >= cumsum[-1] / 2)[0][0]]

# -----------------------------
# WINDOWED FEATURE EXTRACTION
# -----------------------------
rms_vals = []
mdf_vals = []

rms_vals = []
mdf_vals = []

for start in range(0, len(df) - WINDOW + 1, STEP):
    raw_win = df["Raw_EMG"].iloc[start:start+WINDOW].values
    env_win = df["Envelope_EMG"].iloc[start:start+WINDOW].values

    rms_vals.append(rms(env_win))
    mdf_vals.append(mdf(raw_win, FS))
print("Total windows:", len(rms_vals))

rms_vals = np.array(rms_vals)
mdf_vals = np.array(mdf_vals)

# -----------------------------
# NORMALIZE FEATURES
# -----------------------------
rms_n = (rms_vals - rms_vals.mean()) / rms_vals.std()
mdf_n = (mdf_vals - mdf_vals.mean()) / mdf_vals.std()

# -----------------------------
# COMBINE INTO SINGLE VALUE
# -----------------------------
fatigue_value = (ALPHA * rms_n - BETA * mdf_n)

# -----------------------------
# SAVE SINGLE-COLUMN CSV
# -----------------------------
out = pd.DataFrame({
    "Fatigue_Value": fatigue_value.round(4)
})

out.to_csv("data/emg_data_600_combined.csv", index=False)

print("✔ Saved fatigue_single_value.csv")


Total windows: 120205
✔ Saved fatigue_single_value.csv


In [ ]:
import pandas as pd

df = pd.read_csv("data/emg_data_600.csv")
print("Total samples:", len(df))


Total samples: 2402
